# 2.1 Converting MCQ to OSQ

In [72]:
# %pip install python-dotenv openai pandas

import os, re, json, time, logging
from pathlib import Path
from typing import Dict, Any

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

# env + logging
load_dotenv(find_dotenv())
logging.basicConfig(level=logging.DEBUG, format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("mcq2osq")

# paths
CSV_IN = r"C:\Users\rabel\Desktop\dissertation\src\phase1_prep\sysengbench.csv"
ARTIFACTS_DIR      = Path.cwd() / "artifacts_mcq2osq"
RAW_LOGS_DIR       = ARTIFACTS_DIR / "raw_model"
QUESTION_JSON_DIR  = ARTIFACTS_DIR / "question_jsons"
for d in (ARTIFACTS_DIR, RAW_LOGS_DIR, QUESTION_JSON_DIR):
    d.mkdir(parents=True, exist_ok=True)

CSV_OUT   = ARTIFACTS_DIR / "sysengbench_osq.csv"
JSONL_OUT = ARTIFACTS_DIR / "sysengbench_osq.jsonl"

# model/router + run params
MODEL = "openai/gpt-5"             # OpenRouter model id
CONFIDENCE_THRESHOLD = 7           # suitability cut
SAMPLE_N = 5                       # 0 to process all rows
MAX_TOKENS = 3000                 # max tokens for chat completion
TEMPERATURE = 0.0                 # deterministic

client = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
print("OPENROUTER_API_KEY present:", bool(os.getenv("OPENROUTER_API_KEY")))


OPENROUTER_API_KEY present: True


In [73]:
df = pd.read_csv(CSV_IN)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(3))

# Optional: small sample while testing
if SAMPLE_N:
    df = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)
    print("Sampled rows:", len(df))


Rows: 1144
Columns: ['Question ID', 'Tags', 'INCOSE Handbook Category', 'question', 'choiceA', 'choiceB', 'choiceC', 'choiceD', 'answer', 'label', 'Justification']


,Question ID,Tags,INCOSE Handbook Category,question,choiceA,choiceB,choiceC,choiceD,answer,label,Justification
0,1,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,What best describes the concept of uncertainty...,The process of systematically improving and op...,The condition where the outcomes of system fun...,A method for analyzing the costs and benefits ...,The act of integrating different system compon...,B,1,Uncertainty in systems engineering refers to t...
1,2,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,How is risk defined in systems engineering?,The guaranteed outcome of a system's failure.,The process of optimizing a system to avoid an...,The potential for loss or an undesirable outco...,The act of integrating different system compon...,C,2,Risk in systems engineering is conceptualized ...
2,3,Introduction to risk,INCOSEHandbook/Systems Engineering Overview/Sy...,Which of the following best describes the two ...,Uncertainty due to incomplete testing and unce...,Uncertainty due to a lack of knowledge that ca...,Uncertainty due to software errors and uncerta...,Uncertainty due to environmental factors and u...,B,1,Uncertainty in systems engineering can often b...


Sampled rows: 5


In [74]:
def s(x):  # safe string (avoid NaN in prompts)
    return "" if (pd.isna(x) if isinstance(x, float) else x is None) else str(x)

def build_mcq_text(row: pd.Series) -> str:
    """Compact MCQ string for display/LLM (used in classifier context if desired)."""
    return (
        f"Question: {s(row['question'])}\n"
        f"A) {s(row['choiceA'])}\n"
        f"B) {s(row['choiceB'])}\n"
        f"C) {s(row['choiceC'])}\n"
        f"D) {s(row['choiceD'])}\n"
        f"Correct Choice: {s(row['answer'])}"
    )

def build_converter_prompt(row: pd.Series, converter_preamble: str) -> str:
    """Full converter prompt with native columns (no choices list)."""
    return f"""{converter_preamble}

Input MCQ:
Question: {s(row['question'])}
A) {s(row['choiceA'])}
B) {s(row['choiceB'])}
C) {s(row['choiceC'])}
D) {s(row['choiceD'])}
Correct Choice: {s(row['answer'])}
Explanation: {s(row.get('Justification',''))}
"""


In [75]:
CLASSIFIER_PROMPT_TMPL = """You are an educational-assessment expert. Score how suitable a multiple-choice question (MCQ) is for conversion into an open-ended short-answer question (OSQ) **without losing validity**.

SCORE FROM 1–10 (single number). Use these five dimensions to determine the score, but DO NOT output pass/fail or the dimensions themselves:
1) Assessable without showing options.
2) Canonical core answer exists (definition, short derivation, numeric+units, or specific rationale).
3) Removing options won’t add major ambiguity or change the construct being measured.
4) Scope is constrainable in ≤2 sentences; expected answer fits in 1–6 sentences or a numeric expression.
5) Intent and semantics are preserved after conversion.

Rubric anchors (map your final score to the **highest level fully satisfied**):
- 9–10 EXCELLENT: Clear canonical answer; minimal ambiguity; tight scope; construct preserved.
- 7–8 GOOD: Conceptual/application/compare–contrast with crisp boundaries; minor rewording OK.
- 5–6 MARGINAL: Convertible with notable rewording/context; some dependence on options.
- 3–4 POOR: Heavy option dependence or ambiguity; conversion risks construct drift.
- 1–2 UNSUITABLE: No standalone assessment possible; relies on elimination/comparison of options.

Return output ONLY as valid JSON (no extra text):
{
  "suitability_score": <integer 1-10>,
  "justification": "<2–3 sentences referencing the rubric anchors and the most decisive dimensions>"
}

DO NOT add any text outside the JSON object—no explanations, no markdown code fences.
"""

CONVERTER_PROMPT_PREAMBLE = """You convert exactly one multiple-choice question (MCQ) into a precise open-ended short-answer question (OSQ), keeping the construct and difficulty comparable. Assume a prior filter has confirmed this MCQ is suitable for conversion.

Conversion rules (apply all):
1) Remove all multiple-choice artifacts (“which of the following…”, letters A–D, option references). Do not mention options.
2) Preserve the original intent/construct. If the MCQ targeted recall, keep it recall; if numeric or procedural, keep it numeric/procedural. Do not broaden or narrow scope.
3) Keep difficulty approximately the same: if the original expects a fact, definition, derivation, numeric value (with units), or short rationale, require that explicitly.
4) For computations: state givens/assumptions the respondent must use; require units; specify the numeric format (e.g., significant figures or decimal places) and any constants.
5) Produce a canonical expected answer that a knowledgeable grader can match unambiguously. If multiple phrasings or equivalent forms are acceptable, list them in the rubric.
6) Create a detailed rubric that a second grader could apply consistently: point values, criteria for full/partial/no credit, and common acceptable variants and common errors.
7) Identify the Bloom’s taxonomy level that best matches the intended cognitive action of the converted OSQ and justify briefly.

Formatting constraints:
- Be concise, precise, and neutral in tone.
- Do NOT reveal solutions inside the OSQ prompt.
- If numeric: specify required units and rounding (e.g., “Report to 2 decimal places with SI units”).
- If definitional/short rationale: set a length expectation (e.g., “Answer in 1–3 sentences”).
- No extraneous fields or text beyond the JSON. Escape quotes as needed.

Return STRICT JSON with fields:
{
  "osq_prompt": "<final open-ended question users will see>",
  "expected_answer": "<concise canonical answer; if numeric, include units and rounding target>",
  "rubric": {
    "full_credit": "<criteria and points; list acceptable synonyms/representations if any>",
    "partial_credit": "<criteria and points for partially correct work; note common correct elements and typical mistakes>",
    "no_credit": "<criteria for no credit; note disqualifying errors or missing elements>"
  },
  "blooms_level": "<one of: Remember, Understand, Apply, Analyze, Evaluate, Create>",
  "blooms_justification": "<1–2 sentences: identify the cognitive operation elicited by the OSQ prompt and why>"
}

DO NOT add any text outside the JSON object—no explanations, no markdown code fences.
"""


In [76]:
def _clean_to_json_block(text: str) -> str:
    text = re.sub(r"^```(?:json)?\s*", "", (text or "").strip())
    text = re.sub(r"\s*```$", "", text)
    if text.startswith("{") and text.endswith("}"):
        return text
    m = re.search(r"\{.*\}", text, re.S)
    return m.group(0) if m else text

# def chat_json(prompt: str, *, model: str, temperature: float, max_tokens: int,
#               retries: int = 1, raw_path: Path | None = None, tag: str = "") -> Dict[str, Any]:
#     for attempt in range(retries + 1):
#         resp = client.chat.completions.create(
#             model=model,
#             temperature=temperature,
#             messages=[{"role": "user", "content": prompt}],
#             max_tokens=max_tokens,
#         )
#         text = (resp.choices[0].message.content or "").strip()
#         if raw_path: raw_path.write_text(text, encoding="utf-8")

#         log.debug(f"{tag} raw[:180]: {text[:180].replace(chr(10),'\\n')}{'…' if len(text)>180 else ''}")
#         cleaned = _clean_to_json_block(text)
#         log.debug(f"{tag} cleaned[:180]: {cleaned[:180].replace(chr(10),'\\n')}{'…' if len(cleaned)>180 else ''}")

#         try:
#             return json.loads(cleaned)
#         except Exception as e:
#             log.warning(f"{tag} JSON parse failed (try {attempt+1}/{retries+1}): {e}")
#             if attempt < retries:
#                 prompt += "\n\nReminder: Return ONLY valid JSON with no extra text."
#                 time.sleep(0.3)
#             else:
#                 raise ValueError(f"{tag} Model did not return valid JSON:\n{cleaned}") from e


In [77]:
def chat_json(prompt: str, *,
              model: str,
              temperature: float,
              max_tokens: int,
              retries: int = 1,
              raw_path: Path | None = None,
              tag: str = ""
             ) -> tuple[Dict[str, Any], str]:
    """
    Call a chat-completion model and return a tuple:
      (parsed_json_dict, raw_text_from_model)
    - Retries if JSON parsing fails.
    - Optionally saves the raw text to raw_path.
    - Logs debug info with `tag` for traceability.
    """
    for attempt in range(retries + 1):
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
        )
        raw_text = (resp.choices[0].message.content or "").strip()
        if raw_path:
            raw_path.write_text(raw_text, encoding="utf-8")

        log.debug(f"{tag} raw[:180]: {raw_text[:180].replace(chr(10),'\\n')}{'…' if len(raw_text)>180 else ''}")
        cleaned = _clean_to_json_block(raw_text)
        log.debug(f"{tag} cleaned[:180]: {cleaned[:180].replace(chr(10),'\\n')}{'…' if len(cleaned)>180 else ''}")

        try:
            parsed = json.loads(cleaned)
            return parsed, raw_text  # ✅ return both
        except Exception as e:
            log.warning(f"{tag} JSON parse failed (try {attempt+1}/{retries+1}): {e}")
            if attempt < retries:
                # Add a gentle reminder to tighten the next attempt
                prompt += "\n\nReminder: Return ONLY valid JSON with no extra text."
                time.sleep(0.3)
            else:
                raise ValueError(f"{tag} Model did not return valid JSON:\n{cleaned}") from e


In [78]:
from tqdm.notebook import tqdm
from datetime import datetime

DEBUG_FULL = False                 # True = print every row; False = use tqdm bar

def run_classifier(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Classify MCQs for OSQ suitability.
    Logs all prompts and responses to ONE .log file in real time.
    """
    scores, justs, flags = [], [], []

    # One timestamped log file per full run
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    big_log_path = RAW_LOGS_DIR / f"classifier_all_{ts}.log"
    print(f"Writing all raw outputs to: {big_log_path}")

    with open(big_log_path, "a", encoding="utf-8", buffering=1) as biglog:
        iterator = tqdm(df_in.iterrows(),
                        total=len(df_in),
                        desc="Classifying",
                        unit="q",
                        disable=DEBUG_FULL)

        for i, row in iterator:
            qid  = str(row.get("Question ID", i)).strip()
            qtxt = s(row.get("question", ""))  # s() = your sanitizer/cleaner
            prompt = CLASSIFIER_PROMPT_TMPL + "\n\nEvaluation Question:\n" + qtxt

            if DEBUG_FULL:
                print(f"[CLS] id={qid} row={i} len={len(qtxt)}")

            # log prompt for traceability
            biglog.write(f"\n=== Question ID {qid} (row {i}) ===\n")
            biglog.write(f"PROMPT:\n{prompt}\n")
            biglog.flush()

            try:
                parsed, raw_text = chat_json(prompt,
                                            model=MODEL,
                                            temperature=TEMPERATURE,
                                            max_tokens=MAX_TOKENS,
                                            retries=1,
                                            tag=f"[CLS {qid}]")

                score = int(parsed.get("suitability_score", 0))
                just  = parsed.get("justification", "")

                biglog.write("RAW_RESPONSE:\n" + raw_text + "\n")
                biglog.write("PARSED_RESPONSE:\n" + json.dumps(parsed, ensure_ascii=False, indent=2) + "\n")

            except Exception as e:
                score, just = 0, f"Error: {e}"
                biglog.write(f"Error: {e}\n")
            biglog.flush()

            scores.append(score)
            justs.append(just)
            flags.append("Yes" if score >= CONFIDENCE_THRESHOLD else "No")

    out = df_in.copy()
    out["suitability_score"] = scores
    out["confidence_level"] = scores
    out["classification_justification"] = justs
    out["osq_suitable"] = flags
    return out

In [79]:
# ===== Run the classifier with SAMPLE_N respected =====
# Load your DataFrame
df = pd.read_csv(CSV_IN_PATH)   # adjust to your CSV path

# Apply sampling BEFORE running the classifier
if SAMPLE_N and SAMPLE_N > 0:
    df_sample = df.sample(n=min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)
    print(f"Processing SAMPLE_N={SAMPLE_N} rows out of {len(df)}")
else:
    df_sample = df.copy()
    print(f"Processing all {len(df)} rows")

# Run classifier and show a quick preview
df_cls = run_classifier(df_sample)
display(df_cls[["Question ID", "osq_suitable", "suitability_score", "classification_justification"]].head())
print("Suitable:", (df_cls["osq_suitable"] == "Yes").sum(), "/", len(df_cls))

Processing SAMPLE_N=5 rows out of 1144
Writing all raw outputs to: c:\Users\rabel\Desktop\dissertation\src\phase2_conversion\artifacts_mcq2osq\raw_model\classifier_all_20250928_184755.log


Classifying:   0%|          | 0/5 [00:00<?, ?q/s]

18:47:55 | DEBUG | openai._base_client | Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-8b69d413-fa09-4ca4-ba5f-93c081ba76a0', 'json_data': {'messages': [{'role': 'user', 'content': 'You are an educational-assessment expert. Score how suitable a multiple-choice question (MCQ) is for conversion into an open-ended short-answer question (OSQ) **without losing validity**.\n\nSCORE FROM 1–10 (single number). Use these five dimensions to determine the score, but DO NOT output pass/fail or the dimensions themselves:\n1) Assessable without showing options.\n2) Canonical core answer exists (definition, short derivation, numeric+units, or specific rationale).\n3) Removing options won’t add major ambiguity or change the construct being measured.\n4) Scope is constrainable in ≤2 sentences; expected answer fits in 1–6 sentences or a numeric expression.\n5) Intent and semantics are preserved after conversion.\n\nRubric anchor

,Question ID,osq_suitable,suitability_score,classification_justification
0,219,Yes,9,This is essentially a definitional prompt with...
1,810,No,4,MIL-STD-1472G lists multiple valid principles ...
2,502,Yes,9,There is a clear canonical answer: the ICD def...
3,650,Yes,8,The question seeks a canonical acceptance crit...
4,324,Yes,9,There is a clear canonical answer (“by an asso...


Suitable: 4 / 5


In [88]:
df_cls

,Question ID,Tags,INCOSE Handbook Category,question,choiceA,choiceB,choiceC,choiceD,answer,label,Justification,suitability_score,confidence_level,classification_justification,osq_suitable
0,219,Project Management,INCOSEHandbook/Systems Engineering Overview/Sy...,What is the focus of Lean Engineering in syste...,Maximizing financial investment in the system.,Reducing waste and focusing on value-added act...,Accelerating the manufacturing process regardl...,Emphasizing detailed documentation over practi...,B,1,Lean Engineering focuses on reducing waste and...,9,9,This is essentially a definitional prompt with...,Yes
1,810,System Design Requirements,INCOSEHandbook/Specialty Engineering Activitie...,What is the recommended approach for arranging...,Randomly placing them to avoid patterns,Grouping them by function and frequency of use,Arranging them in alphabetical order,Placing them as far apart as possible,B,1,MIL-STD-1472G recommends arranging controls by...,4,4,MIL-STD-1472G lists multiple valid principles ...,No
2,502,NaN,INCOSEHandbook/Lifecycle Stages/Defense Acquis...,What role does the Initial Capabilities Docume...,It serves as the final approval document for s...,It outlines the basic capability needs and ope...,It specifies the detailed technical requiremen...,It provides the cost analysis for the preferre...,B,1,The Initial Capabilities Document (ICD) plays ...,9,9,There is a clear canonical answer: the ICD def...,Yes
3,650,Design Standards,INCOSEHandbook/Technical Management Processes/...,How must shipboard equipment perform during vi...,Equipment should show no signs of physical def...,Equipment must continue to operate correctly d...,Equipment should not detach from its mounting.,All of the above.,D,3,"To pass MIL-STD-167 testing, equipment must no...",8,8,The question seeks a canonical acceptance crit...,Yes
4,324,Use Cases,INCOSEHandbook/Cross-Cutting Systems Engineeri...,How is the interaction between an actor and a ...,Through a generalization relationship,By placing the actor within the system boundary,Via a dependency link indicating reliance on t...,By an association showing the actor can invoke...,D,3,"In SysML, an association between an actor and ...",9,9,There is a clear canonical answer (“by an asso...,Yes


In [86]:
from tqdm.notebook import tqdm  # or tqdm.auto if running as a script

DEBUG_FULL_CONVERT = False   # <-- toggle: True = full per-row debug, False = nice progress bar

def run_converter(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Convert all rows with osq_suitable == 'Yes' to OSQ format.
    Shows either a tqdm progress bar or detailed debug lines.
    """
    rows = []
    total = len(df_in)

    # tqdm will auto-hide if DEBUG_FULL_CONVERT=True
    iterator = tqdm(df_in.iterrows(),
                    total=total,
                    desc="Converting to OSQ",
                    unit="q",
                    disable=DEBUG_FULL_CONVERT)

    for i, row in iterator:
        qid = str(row.get("Question ID", i)).strip()

        if DEBUG_FULL_CONVERT:
            # Detailed debug line for every question
            print(f"[CVT] id={qid} row={i}")

        if row.get("osq_suitable") == "Yes":
            rows.append(convert_one_row(row))
        else:
            rows.append({
                k: "" for k in [
                    "osq_prompt", "expected_answer",
                    "full_credit_criteria", "partial_credit_criteria", "no_credit_criteria",
                    "blooms_level", "blooms_justification"
                ]
            })

    # Merge the new OSQ columns back into the original DataFrame
    return pd.concat([df_in.reset_index(drop=True), pd.DataFrame(rows)], axis=1)


In [84]:
from tqdm.notebook import tqdm
from datetime import datetime

DEBUG_FULL_CONVERT = False   # False = progress bar only, True = print each row too

def run_converter(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Convert all rows with osq_suitable == 'Yes' to OSQ format.
    Logs every prompt/response pair to a single .log file in real time.
    """
    rows = []
    total = len(df_in)

    # Create one log file for the whole conversion run
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    big_log_path = RAW_LOGS_DIR / f"converter_all_{ts}.log"
    print(f"Writing all conversion outputs to: {big_log_path}")

    with open(big_log_path, "a", encoding="utf-8", buffering=1) as biglog:
        iterator = tqdm(df_in.iterrows(),
                        total=total,
                        desc="Converting to OSQ",
                        unit="q",
                        disable=not DEBUG_FULL_CONVERT)

        for i, row in iterator:
            qid = str(row.get("Question ID", i)).strip()

            if DEBUG_FULL_CONVERT:
                print(f"[CVT] id={qid} row={i}")

            if row.get("osq_suitable") == "Yes":
                try:
                    # convert_one_row should now return (parsed_dict, raw_text)
                    parsed, raw_text = convert_one_row(row, return_raw=True)

                    biglog.write(f"\n=== Question ID {qid} (row {i}) ===\n")
                    biglog.write("RAW_RESPONSE:\n" + raw_text + "\n")
                    biglog.write("PARSED_RESPONSE:\n" + json.dumps(parsed, ensure_ascii=False, indent=2) + "\n")

                    rows.append(parsed)
                except Exception as e:
                    biglog.write(f"\n=== Question ID {qid} (row {i}) ===\nError: {e}\n")
                    rows.append({
                        k: "" for k in [
                            "osq_prompt", "expected_answer",
                            "full_credit_criteria", "partial_credit_criteria", "no_credit_criteria",
                            "blooms_level", "blooms_justification"
                        ]
                    })
            else:
                rows.append({
                    k: "" for k in [
                        "osq_prompt", "expected_answer",
                        "full_credit_criteria", "partial_credit_criteria", "no_credit_criteria",
                        "blooms_level", "blooms_justification"
                    ]
                })

            biglog.flush()  # force write after each question

    # Merge all OSQ columns back into the input DataFrame
    return pd.concat([df_in.reset_index(drop=True), pd.DataFrame(rows)], axis=1)


In [87]:
df_cvt = run_converter(df_cls)
display(df_cvt[["Question ID","osq_suitable","suitability_score","osq_prompt","blooms_level"]].head(5))


Converting to OSQ:   0%|          | 0/5 [00:00<?, ?q/s]

18:55:17 | INFO | mcq2osq | [CVT] id=219
18:55:17 | DEBUG | openai._base_client | Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-cc6cd745-12be-413f-8e8c-c83347f0d8ea', 'json_data': {'messages': [{'role': 'user', 'content': 'You convert exactly one multiple-choice question (MCQ) into a precise open-ended short-answer question (OSQ), keeping the construct and difficulty comparable. Assume a prior filter has confirmed this MCQ is suitable for conversion.\n\nConversion rules (apply all):\n1) Remove all multiple-choice artifacts (“which of the following…”, letters A–D, option references). Do not mention options.\n2) Preserve the original intent/construct. If the MCQ targeted recall, keep it recall; if numeric or procedural, keep it numeric/procedural. Do not broaden or narrow scope.\n3) Keep difficulty approximately the same: if the original expects a fact, definition, derivation, numeric value (with units), or short

,Question ID,osq_suitable,suitability_score,osq_prompt,blooms_level
0,219,Yes,9,,
1,810,No,4,,
2,502,Yes,9,,
3,650,Yes,8,,
4,324,Yes,9,,


In [56]:
cols_keep = [
    "Question ID","Tags","INCOSE Handbook Category","question",
    "choiceA","choiceB","choiceC","choiceD","answer","label","Justification",
    "osq_suitable","suitability_score","confidence_level","classification_justification",
    "osq_prompt","expected_answer","full_credit_criteria","partial_credit_criteria",
    "no_credit_criteria","blooms_level","blooms_justification"
]
for c in cols_keep:
    if c not in df_cvt.columns: df_cvt[c] = ""

df_final = df_cvt[cols_keep]
df_final.to_csv(CSV_OUT, index=False)

with open(JSONL_OUT, "w", encoding="utf-8") as f:
    for _, r in df_final.iterrows():
        f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

print("Saved:")
print(" CSV  →", CSV_OUT)
print(" JSONL→", JSONL_OUT)
print("\n=== SUMMARY ===")
print("rows:", len(df_final))
print("suitable:", (df_final["osq_suitable"]=="Yes").sum())
print("avg score:", pd.to_numeric(df_final["suitability_score"], errors="coerce").mean())
display(df_final[["Question ID","osq_suitable","suitability_score","osq_prompt"]].head(3))


Saved:
 CSV  → c:\Users\rabel\Desktop\dissertation\src\phase2_conversion\artifacts_mcq2osq\sysengbench_osq.csv
 JSONL→ c:\Users\rabel\Desktop\dissertation\src\phase2_conversion\artifacts_mcq2osq\sysengbench_osq.jsonl

=== SUMMARY ===
rows: 5
suitable: 5
avg score: 8.8


,Question ID,osq_suitable,suitability_score,osq_prompt
0,219,Yes,9,State the primary focus of Lean Engineering in...
1,810,Yes,7,"According to MIL-STD-1472G, what is the recomm..."
2,502,Yes,9,During the Material Solution Analysis (MSA) ph...


In [57]:
failed = df_final[df_final["osq_suitable"].eq("Yes") & df_final["osq_prompt"].eq("")]
print("Converter failures:", len(failed))
display(failed[["Question ID","question"]].head(10))
print("Raw logs dir:", RAW_LOGS_DIR)
print("Per-question parsed JSON dir:", QUESTION_JSON_DIR)


Converter failures: 0


,Question ID,question


Raw logs dir: c:\Users\rabel\Desktop\dissertation\src\phase2_conversion\artifacts_mcq2osq\raw_model
Per-question parsed JSON dir: c:\Users\rabel\Desktop\dissertation\src\phase2_conversion\artifacts_mcq2osq\question_jsons


In [58]:
bad_ans = df_final[df_final["answer"].isna() | (df_final["answer"].astype(str).str.strip() == "")]
print("Rows with blank 'answer':", len(bad_ans))
display(bad_ans[["Question ID","question","answer","choiceA","choiceB","choiceC","choiceD"]].head(10))


Rows with blank 'answer': 0


,Question ID,question,answer,choiceA,choiceB,choiceC,choiceD
